In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
num_classes = 2
N = 100
X = np.zeros((N*num_classes,2))
T = np.zeros(N*num_classes,dtype=int)
for j in range(num_classes):
    idx = range(j*N,(j+1)*N)
    r = np.linspace(.05,1,N)
    theta = np.linspace(0,2,N)+j*2*np.pi/num_classes+np.random.randn(N)*.1
    X[idx,0] = r*np.cos(theta)
    X[idx,1] = r*np.sin(theta)
    T[idx] = j
x1,x2 = X.T
plt.scatter(x1,x2,c=T,cmap='tab10',vmin=0,vmax=9); plt.axis('equal'); plt.show()
indices = np.arange(T.size)
np.random.shuffle(indices)
train_size = int(.7*T.size)
val_size = int(.15*T.size)
train_indices = indices[:train_size]
val_indices = indices[train_size:train_size+val_size]
test_indices = indices[train_size+val_size:]
X_train = X[train_indices]
T_train = T[train_indices]
Xc_train = np.hstack((np.ones((T_train.size,1)),X_train))
X_val = X[val_indices]
T_val = T[val_indices]
Xc_val = np.hstack((np.ones((T_val.size,1)),X_val))
X_test = X[test_indices]
T_test = T[test_indices]
Xc_test = np.hstack((np.ones((T_test.size,1)),X_test))
def one_hot_encode(labels,num_classes):
    m = labels.shape[0]
    one_hot = np.zeros((m,num_classes))
    one_hot[np.arange(m),labels.flatten()] = 1
    return one_hot
T_train_one_hot = one_hot_encode(T_train,num_classes)
T_val_one_hot = one_hot_encode(T_val,num_classes)

In [ ]:
def ReLU(z):
    d = z*(z>=0)
    return d

def Softmax(x):
    m = x.shape[0]
    P = np.zeros(x.shape)
    for k in range(m):
        z = x[[k],:]-np.max(x[[k],:])
        P[[k],:] = np.exp(z)/np.sum(np.exp(z))
    return P

def FunPolinomio(x,W):
    return np.dot(x,W)

def FunCosto(Preds,T):
    m = T.shape[0]
    return -(1/m)*np.sum(T*np.log(Preds+1e-12))

def FunDerivadas(Preds,T,HiddenActivations,W2,Xc):
    m = T.shape[0]
    DW2 = (1/m)*np.dot(HiddenActivations.T,Preds-T)
    Error = np.dot(Preds-T,W2[1:,:].T)
    dReLU = (HiddenActivations[:,1:]>0).astype(float)
    DW1 = (1/m)*np.dot(Xc.T,Error*dReLU)
    return DW1,DW2

In [ ]:
HiddenNeurons = 8
alfa = 0.1
Iteraciones = 500
np.random.seed(42)
W1 = np.random.randn(3,HiddenNeurons)*np.sqrt(2/2)
W2 = np.random.randn(HiddenNeurons+1,num_classes)*np.sqrt(2/HiddenNeurons)
W1[0,:]=0; W2[0,:]=0
VectorCosto = np.zeros(Iteraciones+1)
VectorCostoVal = np.zeros(Iteraciones+1)
VectorAccuracyTrain = np.zeros(Iteraciones+1)
VectorAccuracyVal = np.zeros(Iteraciones+1)
for It in range(Iteraciones+1):
    z1 = FunPolinomio(Xc_train,W1)
    a1 = ReLU(z1)
    a11 = np.hstack((np.ones((Xc_train.shape[0],1)),a1))
    z2 = FunPolinomio(a11,W2)
    Preds_train = Softmax(z2)
    z1_val = FunPolinomio(Xc_val,W1)
    a1_val = ReLU(z1_val)
    a11_val = np.hstack((np.ones((Xc_val.shape[0],1)),a1_val))
    z2_val = FunPolinomio(a11_val,W2)
    Preds_val = Softmax(z2_val)
    VectorCosto[It] = FunCosto(Preds_train,T_train_one_hot)
    VectorCostoVal[It] = FunCosto(Preds_val,T_val_one_hot)
    VectorAccuracyTrain[It] = np.mean(np.argmax(Preds_train,axis=1)==T_train)*100
    VectorAccuracyVal[It] = np.mean(np.argmax(Preds_val,axis=1)==T_val)*100
    if It < Iteraciones:
        DW1,DW2 = FunDerivadas(Preds_train,T_train_one_hot,a11,W2,Xc_train)
        W1 = W1-alfa*DW1
        W2 = W2-alfa*DW2
print('Training Accuracy:',VectorAccuracyTrain[-1])
print('Validation Accuracy:',VectorAccuracyVal[-1])
plt.plot(VectorCosto,label='Entrenamiento')
plt.plot(VectorCostoVal,label='Validación')
plt.legend()
plt.show()

In [ ]:
x1_prueba = np.linspace(np.min(x1)-.25,np.max(x1)+.25,100)
x2_prueba = np.linspace(np.min(x2)-.25,np.max(x2)+.25,100)
MatrizPred = np.zeros((x2_prueba.size,x1_prueba.size,num_classes))
MatrizFrontera = np.zeros((x2_prueba.size,x1_prueba.size))
for i in range(x1_prueba.size):
    for j in range(x2_prueba.size):
        Xc = np.array([[1,x1_prueba[i],x2_prueba[j]]])
        z1 = FunPolinomio(Xc,W1)
        a1 = ReLU(z1)
        a11 = np.hstack((np.ones((1,1)),a1))
        z2 = FunPolinomio(a11,W2)
        Preds = Softmax(z2)
        MatrizPred[j,i,:] = Preds[0,:]
        MatrizFrontera[j,i] = np.argmax(Preds)
plt.pcolormesh(x1_prueba,x2_prueba,MatrizFrontera,cmap=plt.get_cmap('RdBu',num_classes),shading='auto',vmin=-.5,vmax=num_classes-.5)
plt.scatter(x1,x2,c=T,cmap='RdBu',vmin=-.5,vmax=num_classes-.5,edgecolor='k')
plt.colorbar(ticks=np.arange(num_classes))
plt.show()

In [ ]:
z1_test = FunPolinomio(Xc_test,W1)
a1_test = ReLU(z1_test)
a11_test = np.hstack((np.ones((Xc_test.shape[0],1)),a1_test))
z2_test = FunPolinomio(a11_test,W2)
Preds_test = Softmax(z2_test)
print('Test Accuracy:',np.mean(np.argmax(Preds_test,axis=1)==T_test)*100)

## Actividad
Adapta el ejemplo a cuatro espirales. Completa el número de clases, las neuronas ocultas y las actualizaciones GD.
 Compara 8, 16 y 50 neuronas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
num_classes = 4
N = 150
X = np.zeros((N*num_classes,2))
T = np.zeros(N*num_classes,dtype=int)
for j in range(num_classes):
    idx = range(j*N,(j+1)*N)
    r = np.linspace(.02,1,N)
    theta = np.linspace(0,4,N)+j*2*np.pi/num_classes+np.random.randn(N)*.1
    X[idx,0] = r*np.cos(theta)
    X[idx,1] = r*np.sin(theta)
    T[idx] = j
x1,x2 = X.T
plt.scatter(x1,x2,c=T,cmap='tab10',vmin=0,vmax=9); plt.axis('equal'); plt.show()
indices = np.arange(T.size)
np.random.shuffle(indices)
train_size = int(.7*T.size)
val_size = int(.15*T.size)
train_indices = indices[:train_size]
val_indices = indices[train_size:train_size+val_size]
test_indices = indices[train_size+val_size:]
X_train = X[train_indices]
T_train = T[train_indices]
Xc_train = np.hstack((np.ones((T_train.size,1)),X_train))
X_val = X[val_indices]
T_val = T[val_indices]
Xc_val = np.hstack((np.ones((T_val.size,1)),X_val))
X_test = X[test_indices]
T_test = T[test_indices]
Xc_test = np.hstack((np.ones((T_test.size,1)),X_test))
def one_hot_encode(labels,num_classes):
    m = labels.shape[0]
    one_hot = np.zeros((m,num_classes))
    one_hot[np.arange(m),labels.flatten()] = 1
    return one_hot
T_train_one_hot = one_hot_encode(T_train,num_classes)
T_val_one_hot = one_hot_encode(T_val,num_classes)

In [ ]:
HiddenNeurons = 50
alfa = 0.5
Iteraciones = 3000
np.random.seed(42)
W1 = np.random.randn(3,HiddenNeurons)*np.sqrt(2/2)
W2 = np.random.randn(HiddenNeurons+1,num_classes)*np.sqrt(2/HiddenNeurons)
W1[0,:]=0; W2[0,:]=0
VectorCosto = np.zeros(Iteraciones+1)
VectorCostoVal = np.zeros(Iteraciones+1)
VectorAccuracyTrain = np.zeros(Iteraciones+1)
VectorAccuracyVal = np.zeros(Iteraciones+1)
for It in range(Iteraciones+1):
    z1 = FunPolinomio(Xc_train,W1)
    a1 = ReLU(z1)
    a11 = np.hstack((np.ones((Xc_train.shape[0],1)),a1))
    z2 = FunPolinomio(a11,W2)
    Preds_train = Softmax(z2)
    z1_val = FunPolinomio(Xc_val,W1)
    a1_val = ReLU(z1_val)
    a11_val = np.hstack((np.ones((Xc_val.shape[0],1)),a1_val))
    z2_val = FunPolinomio(a11_val,W2)
    Preds_val = Softmax(z2_val)
    VectorCosto[It] = FunCosto(Preds_train,T_train_one_hot)
    VectorCostoVal[It] = FunCosto(Preds_val,T_val_one_hot)
    VectorAccuracyTrain[It] = np.mean(np.argmax(Preds_train,axis=1)==T_train)*100
    VectorAccuracyVal[It] = np.mean(np.argmax(Preds_val,axis=1)==T_val)*100
    if It < Iteraciones:
        DW1,DW2 = FunDerivadas(Preds_train,T_train_one_hot,a11,W2,Xc_train)
        W1 = W1-alfa*DW1
        W2 = W2-alfa*DW2
print('Training Accuracy:',VectorAccuracyTrain[-1])
print('Validation Accuracy:',VectorAccuracyVal[-1])
plt.plot(VectorCosto,label='Entrenamiento')
plt.plot(VectorCostoVal,label='Validación')
plt.legend()
plt.show()

In [ ]:
x1_prueba = np.linspace(np.min(x1)-.25,np.max(x1)+.25,100)
x2_prueba = np.linspace(np.min(x2)-.25,np.max(x2)+.25,100)
MatrizPred = np.zeros((x2_prueba.size,x1_prueba.size,num_classes))
MatrizFrontera = np.zeros((x2_prueba.size,x1_prueba.size))
for i in range(x1_prueba.size):
    for j in range(x2_prueba.size):
        Xc = np.array([[1,x1_prueba[i],x2_prueba[j]]])
        z1 = FunPolinomio(Xc,W1)
        a1 = ReLU(z1)
        a11 = np.hstack((np.ones((1,1)),a1))
        z2 = FunPolinomio(a11,W2)
        Preds = Softmax(z2)
        MatrizPred[j,i,:] = Preds[0,:]
        MatrizFrontera[j,i] = np.argmax(Preds)
plt.pcolormesh(x1_prueba,x2_prueba,MatrizFrontera,cmap=plt.get_cmap('RdBu',num_classes),shading='auto',vmin=-.5,vmax=num_classes-.5)
plt.scatter(x1,x2,c=T,cmap='RdBu',vmin=-.5,vmax=num_classes-.5,edgecolor='k')
plt.colorbar(ticks=np.arange(num_classes))
plt.show()

In [ ]:
z1_test = FunPolinomio(Xc_test,W1)
a1_test = ReLU(z1_test)
a11_test = np.hstack((np.ones((Xc_test.shape[0],1)),a1_test))
z2_test = FunPolinomio(a11_test,W2)
Preds_test = Softmax(z2_test)
print('Test Accuracy:',np.mean(np.argmax(Preds_test,axis=1)==T_test)*100)